# Validação dos módulos do pipeline ENEM

Este notebook apresenta, de forma curta e reproduzível, as verificações essenciais dos módulos que transformam os microdados do ENEM da camada **Bronze** para a **Silver**.

Ele substitui a parte didática do antigo `05_teste_modulos.ipynb`, que deve ser preservado como histórico de desenvolvimento. Aqui não são repetidas tentativas intermediárias, correções já superadas ou o processamento completo dos 28 anos.

## Objetivos

- validar configuração e catálogo;
- inspecionar os arquivos principais dentro dos ZIPs;
- auditar os cabeçalhos de 1998 a 2025;
- demonstrar o layout único de 1998–2023;
- demonstrar a separação obrigatória entre participantes e resultados em 2024–2025;
- conferir tipagem, destinos e integração ponta a ponta de maneira controlada.


## 1. Preparação do ambiente

O notebook pode ser executado na raiz do projeto ou dentro da pasta `notebook`/`notebooks`. Ele apenas adiciona `src` ao caminho de importação; não modifica os módulos.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


diretorio_atual = Path.cwd().resolve()

if diretorio_atual.name in {"notebook", "notebooks"}:
    raiz_projeto = diretorio_atual.parent
else:
    raiz_projeto = diretorio_atual

src_dir = raiz_projeto / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"Raiz detectada: {raiz_projeto}")
print(f"Código-fonte  : {src_dir}")


Raiz detectada: /home/akel/PycharmProjects/ENEM
Código-fonte  : /home/akel/PycharmProjects/ENEM/src


In [2]:
from enem_pipeline.config import (
    ANO_FINAL,
    ANO_INICIAL,
    BRONZE_DIR,
    CATALOGO_VARIAVEIS,
    CODIFICACAO_MICRODADOS_PADRAO,
    CODIFICACAO_MICRODADOS_POR_ANO,
    SILVER_DIR,
    STAGING_DIR,
    UF_PADRAO,
)


configuracao = pd.Series(
    {
        "Período": f"{ANO_INICIAL}–{ANO_FINAL}",
        "UF padrão": UF_PADRAO,
        "Bronze": str(BRONZE_DIR),
        "Staging": str(STAGING_DIR),
        "Silver": str(SILVER_DIR),
        "Catálogo": str(CATALOGO_VARIAVEIS),
        "Codificação padrão": CODIFICACAO_MICRODADOS_PADRAO,
    },
    name="valor",
)

display(configuracao.to_frame())


,valor
Período,1998–2025
UF padrão,PA
Bronze,/home/akel/PycharmProjects/ENEM/data/bronze
Staging,/home/akel/PycharmProjects/ENEM/data/staging
Silver,/home/akel/PycharmProjects/ENEM/data/silver
Catálogo,/home/akel/PycharmProjects/ENEM/data/metadata/...
Codificação padrão,utf-8


In [3]:
assert BRONZE_DIR.exists(), f"Bronze não encontrada: {BRONZE_DIR}"
assert CATALOGO_VARIAVEIS.exists(), f"Catálogo não encontrado: {CATALOGO_VARIAVEIS}"
assert ANO_INICIAL == 1998
assert ANO_FINAL == 2025

print("Configuração essencial validada.")


Configuração essencial validada.


## 2. Catálogo de variáveis

O catálogo registra a seleção feita para cada edição. A validação abaixo confirma os 28 anos e usa 1998,2003,2008,2014, 2024 e 2025 como pontos representativos.


In [5]:
from enem_pipeline.catalogo import (
    carregar_catalogo,
    obter_variaveis_ano,
)


catalogo = carregar_catalogo()

resumo_catalogo = pd.DataFrame(
    {
        "ano": [1998,2003,2008,2014, 2024, 2025],
        "variaveis_selecionadas": [
            len(obter_variaveis_ano(ano, catalogo))
            for ano in [1998,2003,2008,2014, 2024, 2025]
        ],
        "possui_nu_ano": [
            "NU_ANO" in obter_variaveis_ano(ano, catalogo)
            for ano in [1998,2003,2008,2014, 2024, 2025]
        ],
    }
)

display(resumo_catalogo)


,ano,variaveis_selecionadas,possui_nu_ano
0,1998,25,True
1,2003,33,True
2,2008,34,True
3,2014,53,True
4,2024,52,True
5,2025,52,True


In [6]:
assert catalogo.shape[1] == 28
assert set(map(str, range(ANO_INICIAL, ANO_FINAL + 1))).issubset(catalogo.columns)
assert resumo_catalogo["possui_nu_ano"].all()

print("Catálogo validado para as 28 edições.")


Catálogo validado para as 28 edições.


## 3. Inventário dos ZIPs e reconhecimento dos layouts

Até 2023 existe um arquivo principal `MICRODADOS_ENEM_AAAA.csv`. Em 2024–2025 existem duas bases independentes: `PARTICIPANTES_AAAA.csv` e `RESULTADOS_AAAA.csv`.

Arquivos auxiliares, como `ITENS_PROVA`, não fazem parte da extração principal.


In [8]:
from enem_pipeline.extracao import (
    listar_csvs_ano,
    localizar_zip,
    selecionar_csvs_principais,
)


registros_layout = []

for ano in [1998,2003,2008,2014, 2024, 2025]:
    arquivos = listar_csvs_ano(ano)
    principais = selecionar_csvs_principais(arquivos)

    registros_layout.append(
        {
            "ano": ano,
            "zip": localizar_zip(ano).name,
            "tipos_encontrados": ", ".join(sorted({item["tipo"] for item in arquivos})),
            "arquivos_principais": ", ".join(item["nome_csv"] for item in principais),
            "layout": "unico" if len(principais) == 1 else "separado",
        }
    )

resumo_layout = pd.DataFrame(registros_layout)
display(resumo_layout)


,ano,zip,tipos_encontrados,arquivos_principais,layout
0,1998,microdados_enem_1998.zip,microdados,MICRODADOS_ENEM_1998.csv,unico
1,2003,microdados_enem_2003.zip,microdados,MICRODADOS_ENEM_2003.csv,unico
2,2008,microdados_enem_2008.zip,microdados,MICRODADOS_ENEM_2008.csv,unico
3,2014,microdados_enem_2014.zip,"itens_prova, microdados",MICRODADOS_ENEM_2014.csv,unico
4,2024,microdados_enem_2024.zip,"itens_prova, participantes, resultados","PARTICIPANTES_2024.csv, RESULTADOS_2024.csv",separado
5,2025,microdados_enem_2025.zip,"itens_prova, participantes, resultados","PARTICIPANTES_2025.csv, RESULTADOS_2025.csv",separado


In [9]:
assert resumo_layout.set_index("ano").loc[1998, "layout"] == "unico"
assert resumo_layout.set_index("ano").loc[2003, "layout"] == "unico"
assert resumo_layout.set_index("ano").loc[2008, "layout"] == "unico"
assert resumo_layout.set_index("ano").loc[2014, "layout"] == "unico"
assert resumo_layout.set_index("ano").loc[2024, "layout"] == "separado"
assert resumo_layout.set_index("ano").loc[2025, "layout"] == "separado"

print("Reconhecimento dos layouts validado.")


Reconhecimento dos layouts validado.


## 4. Auditoria do catálogo contra os cabeçalhos

Esta etapa lê somente os cabeçalhos dentro dos ZIPs. Ela não extrai os CSVs completos. O resultado esperado é `OK` para todos os anos.


In [11]:
from enem_pipeline.auditoria import (auditar_todos_anos,
    ler_cabecalho_csv_zip,
)


auditoria_catalogo = auditar_todos_anos(catalogo)

display(
    auditoria_catalogo[
        [
            "ano",
            "variaveis_catalogo",
            "colunas_disponiveis",
            "variaveis_ausentes",
            "status",
        ]
    ]
)


,ano,variaveis_catalogo,colunas_disponiveis,variaveis_ausentes,status
0,1998,25,163,0,OK
1,1999,26,156,0,OK
2,2000,26,154,0,OK
3,2001,33,276,0,OK
4,2002,32,252,0,OK
5,2003,33,222,0,OK
6,2004,33,239,0,OK
7,2005,33,257,0,OK
8,2006,34,258,0,OK
9,2007,34,258,0,OK


In [12]:
problemas_catalogo = auditoria_catalogo.loc[
    auditoria_catalogo["status"] != "OK"
]

assert problemas_catalogo.empty, problemas_catalogo

print("Catálogo compatível com os cabeçalhos de 1998–2025.")


Catálogo compatível com os cabeçalhos de 1998–2025.


## 5. Regras de transformação

Os identificadores e códigos permanecem como texto. Notas e percentuais são convertidos para `DOUBLE`; variáveis categóricas numéricas, para `SMALLINT`.


In [13]:
from enem_pipeline.transformacao import definir_tipo_sql


variaveis_teste = [
    "NU_ANO",
    "TP_FAIXA_ETARIA",
    "TP_SEXO",
    "CO_MUNICIPIO_PROVA",
    "NU_NOTA_CN",
    "TP_STATUS_REDACAO",
]

tipos_teste = pd.DataFrame(
    {
        "variavel": variaveis_teste,
        "tipo_sql": [definir_tipo_sql(item) for item in variaveis_teste],
    }
)

display(tipos_teste)


,variavel,tipo_sql
0,NU_ANO,SMALLINT
1,TP_FAIXA_ETARIA,SMALLINT
2,TP_SEXO,VARCHAR
3,CO_MUNICIPIO_PROVA,VARCHAR
4,NU_NOTA_CN,DOUBLE
5,TP_STATUS_REDACAO,VARCHAR


In [14]:
tipos_esperados = {
    "NU_ANO": "SMALLINT",
    "TP_FAIXA_ETARIA": "SMALLINT",
    "TP_SEXO": "VARCHAR",
    "CO_MUNICIPIO_PROVA": "VARCHAR",
    "NU_NOTA_CN": "DOUBLE",
    "TP_STATUS_REDACAO": "VARCHAR",
}

assert dict(zip(tipos_teste["variavel"], tipos_teste["tipo_sql"])) == tipos_esperados

print("Regras de tipagem validadas.")


Regras de tipagem validadas.


## 6. Estrutura real de 2024–2025

As bases separadas não possuem uma chave individual comum:

- participantes: `NU_INSCRICAO`;
- resultados: `NU_SEQUENCIAL`.

As variáveis compartilhadas são apenas territoriais. Portanto, os dados podem ser combinados posteriormente em nível agregado, por exemplo por município de prova, mas não linha a linha.


In [15]:
def obter_cabecalhos_separados(ano):
    arquivos = selecionar_csvs_principais(listar_csvs_ano(ano))
    return {
        item["tipo"]: set(
            ler_cabecalho_csv_zip(
                ano=ano,
                caminho_interno=item["caminho_interno"],
            )
        )
        for item in arquivos
    }


cabecalhos_2024 = obter_cabecalhos_separados(2024)
cabecalhos_2025 = obter_cabecalhos_separados(2025)

compartilhadas_2024 = sorted(
    cabecalhos_2024["participantes"] & cabecalhos_2024["resultados"]
)
compartilhadas_2025 = sorted(
    cabecalhos_2025["participantes"] & cabecalhos_2025["resultados"]
)

pd.DataFrame(
    {
        "ano": [2024, 2025],
        "colunas_participantes": [len(cabecalhos_2024["participantes"]), len(cabecalhos_2025["participantes"])],
        "colunas_resultados": [len(cabecalhos_2024["resultados"]), len(cabecalhos_2025["resultados"])],
        "compartilhadas": [len(compartilhadas_2024), len(compartilhadas_2025)],
    }
)


,ano,colunas_participantes,colunas_resultados,compartilhadas
0,2024,38,42,5
1,2025,38,70,5


In [16]:
compartilhadas_esperadas = {
    "NU_ANO",
    "CO_MUNICIPIO_PROVA",
    "NO_MUNICIPIO_PROVA",
    "CO_UF_PROVA",
    "SG_UF_PROVA",
}

for ano, cabecalhos, compartilhadas in [
    (2024, cabecalhos_2024, compartilhadas_2024),
    (2025, cabecalhos_2025, compartilhadas_2025),
]:
    assert "NU_INSCRICAO" in cabecalhos["participantes"]
    assert "NU_SEQUENCIAL" in cabecalhos["resultados"]
    assert "NU_INSCRICAO" not in cabecalhos["resultados"]
    assert "NU_SEQUENCIAL" not in cabecalhos["participantes"]
    assert set(compartilhadas) == compartilhadas_esperadas

print("Separação metodológica de 2024–2025 validada.")


Separação metodológica de 2024–2025 validada.


## 7. Destinos Silver esperados

O layout único produz um Parquet anual. O layout separado produz dois Parquets anuais independentes.


In [17]:
from enem_pipeline.gravacao import (
    construir_caminho_parquet,
    construir_caminho_parquet_separado,
)


destinos_exemplo = pd.DataFrame(
    {
        "ano": [2003, 2024, 2024],
        "base": ["microdados", "participantes", "resultados"],
        "destino": [
            construir_caminho_parquet(2003, UF_PADRAO),
            construir_caminho_parquet_separado(2024, UF_PADRAO, "participantes"),
            construir_caminho_parquet_separado(2024, UF_PADRAO, "resultados"),
        ],
    }
)

display(destinos_exemplo)


,ano,base,destino
0,2003,microdados,/home/akel/PycharmProjects/ENEM/data/silver/mi...
1,2024,participantes,/home/akel/PycharmProjects/ENEM/data/silver/pa...
2,2024,resultados,/home/akel/PycharmProjects/ENEM/data/silver/re...


## 8. Testes de integração opcionais

As células seguintes ficam desativadas por padrão porque podem extrair arquivos grandes e criar Silver caso elas ainda não existam. Quando as Silver já existem, `processar_ano` apenas as reutiliza e valida.

Altere as flags conscientemente para executar os testes ponta a ponta.


In [18]:
EXECUTAR_TESTE_LAYOUT_UNICO = False
EXECUTAR_TESTE_LAYOUT_SEPARADO = False

print("Teste layout único   :", EXECUTAR_TESTE_LAYOUT_UNICO)
print("Teste layout separado:", EXECUTAR_TESTE_LAYOUT_SEPARADO)


Teste layout único   : False
Teste layout separado: False


In [19]:
from enem_pipeline.processamento import processar_ano


resultado_2003 = None
resultado_2024 = None

if EXECUTAR_TESTE_LAYOUT_UNICO:
    resultado_2003 = processar_ano(
        ano=2003,
        uf=UF_PADRAO,
        catalogo=catalogo,
    )
    display(resultado_2003)

if EXECUTAR_TESTE_LAYOUT_SEPARADO:
    resultado_2024 = processar_ano(
        ano=2024,
        uf=UF_PADRAO,
        catalogo=catalogo,
    )
    display(resultado_2024)

if not any([EXECUTAR_TESTE_LAYOUT_UNICO, EXECUTAR_TESTE_LAYOUT_SEPARADO]):
    print("Testes de integração não executados.")


Testes de integração não executados.


In [20]:
if resultado_2003 is not None:
    assert resultado_2003["layout"] == "unico"
    assert resultado_2003["duplicacoes"] == 0
    assert resultado_2003["outras_ufs"] == 0
    assert resultado_2003["anos_invalidos"] == 0

if resultado_2024 is not None:
    assert resultado_2024["layout"] == "separado"
    assert resultado_2024["participantes"]["duplicacoes"] == 0
    assert resultado_2024["resultados"]["duplicacoes"] == 0
    assert resultado_2024["correspondencia_municipal"]["municipios_com_diferenca"] == 0

print("Validações opcionais concluídas para os testes executados.")


Validações opcionais concluídas para os testes executados.


## Conclusão

Os testes essenciais ficam concentrados neste notebook. O processamento completo das 28 edições, a limpeza do staging e a geração do manifesto pertencem ao notebook `06_pipeline_bronze_silver.ipynb`.

O antigo `05_teste_modulos.ipynb` deve permanecer arquivado como registro das decisões e correções realizadas durante o desenvolvimento.
